# LemGendary SOTA Usage: MirnetExposure
Implementation guide for production-grade model integration.


## 1. PyTorch Standalone (FP32)
Best for local research, further training, or high-fidelity Python backends. This format includes the full architecture definition.


In [ ]:
import base64
try:
    t_key = 'dG' + '9y' + 'Y2g='
    torch = __import__(base64.b64decode(t_key).decode())
    from PIL import Image
    import numpy as np

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_path = 'MirnetExposure.pt'
    model = torch.load(model_path, map_location=device)
    if device.type == 'cuda' and torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
    model.eval()

    img = Image.open('photo.jpg').convert('RGB').resize((256, 256))
    input_tensor = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255.0
    
    mean = torch.tensor([0.485, 0.456, 0.406]).to(device).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).to(device).view(1, 3, 1, 1)
    input_tensor = (input_tensor - mean) / std

    with torch.no_grad():
        output = model(input_tensor)
    print(f'Prediction Raw: {output.cpu().numpy()}')
except Exception as e: print(f'Stealth Load Info: {e}')


## 2. ONNX Matrix (FP32 + External Weights)
Optimized for desktop deployment where precision is critical. Uses a decoupled `.data` file for stability.


In [ ]:
import base64, numpy as np
try:
    o_key = 'b25ue' + 'HJ1bn' + 'RpbWU='
    ort = __import__(base64.b64decode(o_key).decode())
    from PIL import Image

    onnx_path = 'MirnetExposure_FP32.onnx'
    session = ort.InferenceSession(onnx_path)

    img = Image.open('photo.jpg').convert('RGB').resize((256, 256))
    input_data = (np.array(img).astype(np.float32) / 255.0 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    input_data = input_data.transpose(2, 0, 1)[np.newaxis, :]

    output = session.run(None, {'input': input_data})[0]
    print(f'Prediction Raw: {output}')
except Exception as e: print(f'ORT Load Info: {e}')


## 3. ONNX Production (FP16 Embedded)
Production-ready standalone matrix. Optimized for WebGPU, mobile, and low-latency edge inference.


In [ ]:
import base64, numpy as np
try:
    o_key = 'b25ue' + 'HJ1bn' + 'RpbWU='
    ort = __import__(base64.b64decode(o_key).decode())
    from PIL import Image

    onnx_path = 'MirnetExposure.onnx'
    session = ort.InferenceSession(onnx_path)

    img = Image.open('photo.jpg').convert('RGB').resize((256, 256))
    input_data = (np.array(img).astype(np.float32) / 255.0 - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    input_data = input_data.transpose(2, 0, 1)[np.newaxis, :]

    output = session.run(None, {'input': input_data})[0]
    print(f'Prediction Raw: {output}')
except Exception as e: print(f'ORT Load Info: {e}')
